# Experimento pareado: selección por F1 frente a F2 en LSTM32 dropout 0,2 — Colab CPU

Se comparan dos entrenamientos desde cero con los mismos datos, arquitectura, semilla, BCE, Adam, learning rate y regularización. Solo cambia el objetivo de selección: F2 frente a F1 para elegir checkpoint y umbral. Ambas corridas usan CPU en la misma sesión; el resultado se interpreta como una comparación CPU y no se mezcla directamente con corridas GPU anteriores. El test 2025 permanece reservado.


## 1. Subir y verificar el paquete CPU

El paquete ya está listo como `runs/lstm_colab_f1_cpu_v1.zip`. Para regenerarlo desde la raíz del repositorio, ejecuta `python3.13 scripts/empaquetar_lstm_f1.py --reference-config configs/experimento_lstm32_f1_ref_cpu_v1.json --candidate-config configs/experimento_lstm32_f1_candidate_cpu_v1.json --output runs/lstm_colab_f1_cpu_v1.zip`. Sube el ZIP al panel **Archivos** de Colab cuando la primera celda lo solicite. El SHA-256 esperado está fijado en el notebook; no lo reemplaces con otro paquete.


In [ ]:
from pathlib import Path
import hashlib, json, stat, subprocess, sys, zipfile
from google.colab import files

ROOT = Path('/content/f1-lstm-f1-cpu-v1')
PACKAGE = Path('/content/lstm_colab_f1_cpu_v1.zip')
EXPECTED_PACKAGE_SHA256 = '2cdffa3ba506fdd829637ef91a439b6e979f09ee970bced0ca64a56916edfacd'
if ROOT.exists(): raise FileExistsError('Usa una sesión limpia: /content/f1-lstm-f1-cpu-v1 ya existe.')
if not PACKAGE.is_file():
    uploaded = files.upload()
    if len(uploaded) != 1: raise ValueError('Sube exactamente el ZIP CPU del par LSTM F1/F2.')
    name, content = next(iter(uploaded.items()))
    if Path(name).name != 'lstm_colab_f1_cpu_v1.zip': raise ValueError('Se requiere lstm_colab_f1_cpu_v1.zip; no uses el ZIP GPU ni otro archivo.')
    PACKAGE.write_bytes(content)
def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024), b''): h.update(chunk)
    return h.hexdigest()
package_hash = sha256(PACKAGE)
print('SHA-256 del paquete:', package_hash)
if package_hash != EXPECTED_PACKAGE_SHA256: raise ValueError('El hash no coincide con el ZIP CPU corregido.')
with zipfile.ZipFile(PACKAGE) as z:
    members=z.infolist(); names=set()
    for m in members:
        p=Path(m.filename)
        if p.is_absolute() or '..' in p.parts or '\\' in m.filename or stat.S_ISLNK(m.external_attr >> 16): raise ValueError(f'Ruta ZIP no segura: {m.filename}')
        if m.filename in names: raise ValueError(f'Miembro duplicado: {m.filename}')
        names.add(m.filename)
    manifest=json.loads(z.read('colab_package_manifest.json'))
    if manifest.get('artifact_version') != 'lstm_colab_f1_cpu_inputs_v1' or manifest.get('test_arrays_included') is not False: raise ValueError('Contrato de paquete no admitido o contiene test.')
    expected=set(manifest['files_sha256']) | {'colab_package_manifest.json'}
    if names != expected: raise ValueError('Los archivos del ZIP no coinciden con su manifiesto.')
    ROOT.mkdir(parents=True)
    z.extractall(ROOT)
for name, digest in manifest['files_sha256'].items():
    if sha256(ROOT/name) != digest: raise ValueError(f'SHA-256 incorrecto: {name}')
print('Manifiesto verificado:', len(manifest['files_sha256']), 'archivos; sin datos de prueba.')


## 2. Instalar dependencias y confirmar runtime CPU

Esta variante fija `device=cpu` en ambas configuraciones. No necesita GPU/CUDA; aunque Colab muestre una GPU disponible, el entrenamiento seguirá ejecutándose en CPU. La corrida puede tardar más que la pareja GPU.


In [ ]:
import subprocess, sys
from pathlib import Path
ROOT = Path('/content/f1-lstm-f1-cpu-v1')
if not (ROOT / 'requirements-colab-experiments.txt').is_file():
    raise RuntimeError('No encuentro el paquete CPU extraído en /content/f1-lstm-f1-cpu-v1. Ejecuta primero la sección 1 desde una sesión Colab activa.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT/'requirements-colab-experiments.txt')], check=True)
import torch
print('PyTorch:', torch.__version__, '| CUDA disponible:', torch.cuda.is_available(), '| dispositivo efectivo de ambas corridas: CPU')


## 3. Entrenar la pareja en CPU

La referencia selecciona checkpoint y umbral por F2; la candidata lo hace por F1. Ambas usan early stopping en validación a umbral fijo 0,5, paciencia seis y desempate por BCE menor. El runner valida que solo cambie el objetivo de selección y que ambas corridas compartan CPU y entorno. Si falla, la celda imprime `status.json`. La validación es de desarrollo, no prueba independiente.


In [ ]:
import json, subprocess, sys
from pathlib import Path
from google.colab import files
ROOT = Path('/content/f1-lstm-f1-cpu-v1')
if not (ROOT / 'scripts/entrenar_comparacion_lstm_f1.py').is_file():
    raise RuntimeError('No encuentro el runner corregido en el paquete v2. Ejecuta primero la sección 1 desde una sesión Colab activa.')
RUN = ROOT/'runs/lstm32_f1_objective_cpu_v1_seed42'
if RUN.exists() or RUN.with_suffix('.zip').exists(): raise FileExistsError('La salida ya existe. Conserva los artefactos y usa una sesión/salida nueva.')
result = subprocess.run(
    [sys.executable, 'scripts/entrenar_comparacion_lstm_f1.py', '--reference-config', 'configs/experimento_lstm32_f1_ref_cpu_v1.json', '--candidate-config', 'configs/experimento_lstm32_f1_candidate_cpu_v1.json', '--output', str(RUN.relative_to(ROOT))],
    cwd=ROOT,
    check=False,
)
if result.returncode != 0:
    status_path = RUN/'status.json'
    if status_path.is_file():
        print('Estado del par:')
        print(json.dumps(json.loads(status_path.read_text()), indent=2))
    raise RuntimeError(f'La corrida falló con código {result.returncode}; revisa el estado anterior antes de reintentar.')
summary=json.loads((RUN/'pair_summary.json').read_text())
print(json.dumps(summary, indent=2))
files.download(str(RUN.with_suffix('.zip')))


## 4. Devolver resultados al proyecto

Descarga `lstm32_f1_objective_cpu_v1_seed42.zip` desde el selector de archivos de Colab y guárdalo intacto en `runs/` del repositorio local. La evaluación local verificará hashes y paridad CPU. Anota que esta pareja se ejecutó en CPU y compárala dentro de la pareja; no la trates como réplica exacta de corridas previas en GPU. El test 2025 no se utilizó.
